In [16]:
import os
import pandas as pd
from openai import OpenAI

import numpy as np
import json
from dotenv import load_dotenv

In [17]:
repo_dir = "/Users/haya1/Documents/LanguageModel_Labels/congressional_bills"
os.chdir(repo_dir)

# Place API_KEY in the .env file
load_dotenv()
api_key = os.environ.get('API_KEY')
client = OpenAI(api_key=api_key)

## Generate prompts

In [18]:
data_dir = os.path.join(repo_dir, "01_bills")
llm_dir = os.path.join(repo_dir, "02_llm")
base_prompt = open(os.path.join(llm_dir, "base_prompt_json.txt"), 'r').read()
print("Loaded base question")
# print(base_prompt)

Loaded base question


In [165]:
prompting_strategies = pd.DataFrame(data={
    "Name": [
        "No Modification", 
        "Persona Modification", "Persona Modification", "Persona Modification", "Persona Modification", 
        "Chain-of-Thoughts Prompting", "Chain-of-Thoughts Prompting", "Chain-of-Thoughts Prompting", 
        "Few-Shot Prompting", 
        # "Few-Shot Prompting", "Few-Shot Prompting"
        ],
    "BeforeQuestion": [
        "",
        "You are a knowledgeable political analyst. ",
        "Answer this question as if you are a political scientist that studies legislation in the United States Congress. ",
        "Answer this question as if you are an expert in United States politics. ",
        "Answer this question as if you were a helpful research assistant for a political scientist. ",
        "", "", "",
        "", 
        # "", "",
        ],
    "BeforeAnswer": [
        "", 
        "", "", "", "", 
        "Think carefully. ", 
        "Let's think step by step. Lay out each step. ", 
        "Please provide an explanation for your answer. ",
        "", 
        # "", "",
    ],
    "Explanation":[
        "", 
        "", "", "", "",
        ',\n    "explanation": a one-sentence explanation of your bill category answer',
        ',\n    "explanation": a one-sentence explanation of your bill category answer',
        ',\n    "explanation": a one-sentence explanation of your bill category answer',
        "", 
        # "", "",
    ],
    "Model":[
        "gpt-3.5-turbo",
        "gpt-3.5-turbo", "gpt-3.5-turbo", "gpt-3.5-turbo", "gpt-3.5-turbo", 
        "gpt-3.5-turbo", "gpt-3.5-turbo", "gpt-3.5-turbo", 
        "ft:gpt-3.5-turbo-0125:massachusetts-institute-of-technology:recipe-ner:9l4sPEat",
        # "ft:gpt-3.5-turbo-0125:massachusetts-institute-of-technology:recipe-ner:9l4sPEat",
        # "ft:gpt-3.5-turbo-0125:massachusetts-institute-of-technology:recipe-ner:9l4sPEat"
    ]
})

In [103]:
# # Bills used for few-shot prompting
# bills = pd.read_csv(os.path.join(data_dir, "bills.csv")).groupby('Major').sample(n=3) # TODO: remove .groupby('Major').sample(n=1)
# bills = bills[["BillID", "Major", "Description"]] 

# bills_train = bills[0::3]
# print(f"Number of bills used as examples for few-shot training = {len(bills_train)}")
# bills_val = bills[1::3]
# print(f"Number of bills used as examples for few-shot validation = {len(bills_val)}")

# bills_train.to_csv(os.path.join(llm_dir, "bills_train.csv"), index=False)
# bills_val.to_csv(os.path.join(llm_dir, "bills_val.csv"), index=False)

Number of bills used as examples for few-shot training = 21
Number of bills used as examples for few-shot validation = 21
Number of bills used for few-shot testing = 21


In [120]:
# import json

# def prepare_example(bill):
#     messages = []

#     example_prompt = base_prompt % ("", bill["Description"], "", "")
#     messages.append({"role": "user", "content": example_prompt})

#     example_response = "{\"category\": %d, \"confidence\": %.2f}" % (bill["Major"], 1.00)
#     messages.append({"role": "assistant", "content": example_response})

#     return {"messages": messages}

# def save_examples(josnl_path, examples):
#     with open(josnl_path, "w") as f:
#         for example in examples:
#             f.write(json.dumps(example) + "\n")

# bills_train_path = os.path.join(llm_dir, "bills_train.jsonl")
# bills_val_path = os.path.join(llm_dir, "bills_val.jsonl")
# save_examples(bills_train_path, bills_train.apply(prepare_example, axis=1).tolist())
# save_examples(bills_val_path, bills_val.apply(prepare_example, axis=1).tolist())

# # !head -5 "02_llm/bills_train.jsonl"

{"messages": [{"role": "user", "content": "Here is a description of a bill introduced in the U.S. Congress:\n\"To amend the Internal Revenue Code to disallow a deduction for interest paid or accrued on late paid taxes..\"\n\nPlease classify this description into one of the following categories:\n1. Macroeconomics\n2. Civil Rights, Minority Issues, and Civil Liberties\n3. Health\n4. Agriculture\n5. Labor and Employment\n6. Education\n7. Environment\n8. Energy\n9. Immigration\n10. Transportation\n11. Law, Crime, and Family Issues\n12. Social Welfare\n13. Community Development and Housing Issues\n14. Banking, Finance, and Domestic Commerce\n15. Defense\n16. Space, Science, Technology, and Communications\n17. Foreign Trade\n18. International Affairs and Foreign Aid\n19. Government Operations\n20. Public Lands and Water Management\n21. Arts and Entertainment\n\nOutput a JSON object structured like: {\n    \"category\": an integer from 1 to 21 of the bill category,\n    \"confidence\": a num

In [186]:
# Exclude bills used for prompting from bills data
bills_train = pd.read_csv(os.path.join(llm_dir, "bills_train.csv"))
bills_val= pd.read_csv(os.path.join(llm_dir, "bills_val.csv"))
bills_exclude = pd.concat([bills_train, bills_val]).reset_index()
print(f"Number of bills used for few-shot model training and validation = {len(bills_exclude)}")

bills = pd.read_csv(os.path.join(data_dir, "bills.csv"))
condition = bills["BillID"].apply(lambda x: x not in bills_exclude["BillID"].to_list())
bills = bills[condition].groupby('Major').sample(n=2) # TODO: remove .groupby('Major').sample(n=2) 
print(f"Number of bills used for testing for prompting strategies = {len(bills)}")


Number of bills used for few-shot model training and validation = 42
Number of bills used for testing for prompting strategies = 42


In [121]:
# with open(bills_train_path, "rb") as f:
#     response_train = client.files.create(file=f, purpose="fine-tune")

# with open(bills_val_path, "rb") as f:
#     response_val = client.files.create(file=f, purpose="fine-tune")

# print("Training file ID:",  response_train.id)
# print("Validation file ID:",  response_val.id)

# response = client.fine_tuning.jobs.create(
#     training_file= response_train.id,
#     validation_file=response_val.id,
#     model="gpt-3.5-turbo",
#     suffix="recipe-ner",
# )

# job_id = response.id

# print("Job ID:", response.id)
# print("Status:", response.status)

Training file ID: file-JsIL2oQ36Qd36DQuba1lwSBR
Validation file ID: file-SK5ykfcqDhmdIIaTQqNGiCQx
Job ID: ftjob-E30Cz63sprmK1iWHUq8PP8Uf
Status: validating_files


In [141]:
# response = client.fine_tuning.jobs.retrieve(job_id)

# print("Job ID:", response.id)
# print("Status:", response.status)
# print("Trained Tokens:", response.trained_tokens)

Job ID: ftjob-E30Cz63sprmK1iWHUq8PP8Uf
Status: running
Trained Tokens: None


In [146]:
# response = client.fine_tuning.jobs.list_events(job_id)

# events = response.data
# events.reverse()

# for event in events:
#     print(event.message)

Step 69/84: training loss=0.00, validation loss=0.91
Step 70/84: training loss=0.00, validation loss=0.00
Step 71/84: training loss=0.00, validation loss=0.00
Step 72/84: training loss=0.00, validation loss=0.00
Step 73/84: training loss=0.00, validation loss=0.00
Step 74/84: training loss=0.01, validation loss=0.47
Step 75/84: training loss=0.00, validation loss=0.00
Step 76/84: training loss=0.00, validation loss=0.00
Step 77/84: training loss=0.00, validation loss=1.14
Step 78/84: training loss=0.00, validation loss=0.00
Step 79/84: training loss=0.02, validation loss=0.00
Step 80/84: training loss=0.00, validation loss=0.00
Step 81/84: training loss=0.00, validation loss=0.00
Step 82/84: training loss=0.00, validation loss=1.58
Step 83/84: training loss=0.00, validation loss=0.00
Step 84/84: training loss=0.00, validation loss=0.00, full validation loss=0.20
Checkpoint created at step 42 with Snapshot ID: ft:gpt-3.5-turbo-0125:massachusetts-institute-of-technology:recipe-ner:9l4sPW

In [147]:
# # Step 84/84: training loss=0.00, validation loss=0.00, full validation loss=0.20
# # Checkpoint created at step 42 with Snapshot ID: ft:gpt-3.5-turbo-0125:massachusetts-institute-of-technology:recipe-ner:9l4sPWhF:ckpt-step-42
# # Checkpoint created at step 63 with Snapshot ID: ft:gpt-3.5-turbo-0125:massachusetts-institute-of-technology:recipe-ner:9l4sPS4H:ckpt-step-63
# # New fine-tuned model created: ft:gpt-3.5-turbo-0125:massachusetts-institute-of-technology:recipe-ner:9l4sPEat
# # The job has successfully completed

# response = client.fine_tuning.jobs.retrieve(job_id)
# fine_tuned_model_id = response.fine_tuned_model

# if fine_tuned_model_id is None: 
#     raise RuntimeError("Fine-tuned model ID not found. Your job has likely not been completed yet.")

# print("Fine-tuned model ID:", fine_tuned_model_id)

Fine-tuned model ID: ft:gpt-3.5-turbo-0125:massachusetts-institute-of-technology:recipe-ner:9l4sPEat


In [187]:
bill_ids = []
strategies = []
prompts = []
models = []

for _, bill in bills.iterrows():
    for _, strategy in prompting_strategies.iterrows():
        bill_ids.append(bill["BillID"])
        strategies.append(strategy["Name"])
        prompts.append(base_prompt % (strategy["BeforeQuestion"], bill["Description"], strategy["BeforeAnswer"], strategy["Explanation"]))
        models.append(strategy["Model"])

prompts = pd.DataFrame(data = {
    "PromptID": list(range(1,len(prompts)+1)),
    "BillID": bill_ids,
    "PromptingStrategy": strategies,
    "Prompt": prompts,
    "Model": models
})
prompts.to_csv(os.path.join(llm_dir, "prompts.csv"), index=False)

## Generate responses

In [188]:
# SYSTEM_PROMPT = ""

# define a response function that gives us the LLM's response to a user prompt
def query_llm(prompt, model, temperature=0, num_responses=1):
    response_text = client.chat.completions.create(
        model = model,
        logprobs = True,
        n = num_responses,
        temperature = temperature,
        response_format={"type": "json_object"},
        messages=[
            # {"role": "system",  #  role (either "system", "user", or "assistant") The system message helps set the behavior of the assistant. For example, you can modify the personality of the assistant or provide specific instructions about how it should behave throughout the conversation. However note that the system message is optional and the model’s behavior without a system message is likely to be similar to using a generic message such as "You are a helpful assistant."
            #  "content": system},
            {"role": "user", 
            "content": prompt},
        ]
    ).choices[0].message.content
    
    response_json = json.loads(response_text)
    
    out = [np.nan, np.nan, np.nan] # category, confidence, explanation

    if "category" in response_json.keys():
        out[0] = response_json["category"]

    if "confidence" in response_json.keys():
        out[1] = response_json["confidence"]

    if "explanation" in response_json.keys():
        out[2] = response_json["explanation"]
    
    return out

In [189]:
prompts = pd.read_csv(os.path.join(llm_dir, "prompts.csv"))
print(f"Loaded prompts data, n = {len(prompts)}")

prompt_id = []
bill_id = []
major_llm = []
confidence_llm = []
explanation_llm = []

for i in range(len(prompts)):
    prompt_id.append(prompts["PromptID"][i])
    bill_id.append(prompts["BillID"][i])
    
    response = query_llm(prompt=prompts["Prompt"][i], 
                         model=prompts["Model"][i], 
                         temperature=0)
    
    major_llm.append(response[0])
    confidence_llm.append(response[1])
    explanation_llm.append(response[2])

responses = pd.DataFrame(data = {
    "PromptID": prompt_id,
    "MajorLLM": major_llm,
    "ConfidenceLLM": confidence_llm,
    "ExplanationLLM": explanation_llm
})

Loaded prompts data, n = 378


In [190]:
major_text_ours = {
    1 : "Macroeconomics",
    2 : "Civil Rights, Minority Issues, and Civil Liberties",
    3 : "Health",
    4 : "Agriculture",
    5 : "Labor and Employment",
    6 : "Education",
    7 : "Environment",
    8 : "Energy",
    9 : "Immigration",
    10: "Transportation",
    11: "Law, Crime, and Family Issues",
    12: "Social Welfare",
    13: "Community Development and Housing Issues",
    14: "Banking, Finance, and Domestic Commerce",
    15: "Defense",
    16: "Space, Science, Technology, and Communications",
    17: "Foreign Trade",
    18: "International Affairs and Foreign Aid",
    19: "Government Operations",
    20: "Public Lands and Water Management",
    21: "Arts and Entertainment"
}

responses["MajorTextLLM"] = responses["MajorLLM"].apply(lambda x: major_text_ours[x] if pd.notnull(x) else x)
responses = responses[["PromptID", "MajorLLM", "MajorTextLLM", "ConfidenceLLM", "ExplanationLLM"]]

print(responses.head())

responses.to_csv(os.path.join(llm_dir, "responses.csv"), index=False)

   PromptID  MajorLLM                             MajorTextLLM  ConfidenceLLM  \
0         1        14  Banking, Finance, and Domestic Commerce           0.85   
1         2        14  Banking, Finance, and Domestic Commerce           0.85   
2         3        14  Banking, Finance, and Domestic Commerce           0.85   
3         4        14  Banking, Finance, and Domestic Commerce           0.85   
4         5        14  Banking, Finance, and Domestic Commerce           0.85   

  ExplanationLLM  
0            NaN  
1            NaN  
2            NaN  
3            NaN  
4            NaN  


In [191]:
bills = pd.read_csv(os.path.join(data_dir, "bills.csv"))[['BillID', 'Major', 'MajorText', 'Description']]
prompts = pd.read_csv(os.path.join(llm_dir, "prompts.csv"))
responses = pd.read_csv(os.path.join(llm_dir, "responses.csv"))

condition = bills["BillID"].apply(lambda x: x in prompts["BillID"].unique())
bills = bills[condition]

df = prompts.merge(responses,  on="PromptID", validate="one_to_one")
df = df.merge(bills, on="BillID", validate="many_to_one")
df = df[["PromptID", "BillID", "PromptingStrategy", "Prompt", "Description", "Major", "MajorText", "MajorLLM", "MajorTextLLM", "ConfidenceLLM", "ExplanationLLM"]]
df.head()

,PromptID,BillID,PromptingStrategy,Prompt,Description,Major,MajorText,MajorLLM,MajorTextLLM,ConfidenceLLM,ExplanationLLM
0,1,110-S-2885,No Modification,Here is a description of a bill introduced in ...,A bill to amend the Internal Revenue Code of 1...,1,Macroeconomics,14,"Banking, Finance, and Domestic Commerce",0.85,NaN
1,2,110-S-2885,Persona Modification,You are a knowledgeable political analyst. Her...,A bill to amend the Internal Revenue Code of 1...,1,Macroeconomics,14,"Banking, Finance, and Domestic Commerce",0.85,NaN
2,3,110-S-2885,Persona Modification,Answer this question as if you are a political...,A bill to amend the Internal Revenue Code of 1...,1,Macroeconomics,14,"Banking, Finance, and Domestic Commerce",0.85,NaN
3,4,110-S-2885,Persona Modification,Answer this question as if you are an expert i...,A bill to amend the Internal Revenue Code of 1...,1,Macroeconomics,14,"Banking, Finance, and Domestic Commerce",0.85,NaN
4,5,110-S-2885,Persona Modification,Answer this question as if you were a helpful ...,A bill to amend the Internal Revenue Code of 1...,1,Macroeconomics,14,"Banking, Finance, and Domestic Commerce",0.85,NaN


In [192]:
df.to_csv(os.path.join(llm_dir, "prompts_and_responses.csv"), index=False)